# Joan Tryhard

### Imports

In [1]:
import pandas as pd
import sklearn
import imblearn
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

### Get Data and Preprocess

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import pandas as pd

TESTING_WITH_TRAIN_DATA = False
TESTING_WITH_KAGGLE_DATA = True


train = pd.read_csv("data/train_dataset_processed.csv")
test = pd.read_csv("data/test_dataset_processed.csv")

languages = train['language'].unique()


if TESTING_WITH_TRAIN_DATA:

    # Split into train and test sets
    sentences = train[['language', 'sentence_id']].drop_duplicates()
    sentences_train, sentences_test = train_test_split(sentences,test_size=0.2,random_state=42)
    train_set = pd.merge(train, sentences_train, on=['language', 'sentence_id'])
    test_set = pd.merge(train, sentences_test, on=['language', 'sentence_id'])

    # One-hot encode 'language' in train and test sets with language_LANGUAGE
    enc = OneHotEncoder(sparse_output=True)
    language_encoded = enc.fit_transform(train_set[['language']])
    language_df = pd.DataFrame(language_encoded.toarray(),
                            columns=enc.get_feature_names_out(['language']),
                            index=train_set.index)
    train = pd.concat([train_set.drop(columns=['language']), language_df], axis=1)

    # Same for test set
    language_encoded_test = enc.transform(test_set[['language']])
    language_df_test = pd.DataFrame(language_encoded_test.toarray(),
                                    columns=enc.get_feature_names_out(['language']),
                                    index=test_set.index)
    test = pd.concat([test_set.drop(columns=['language']), language_df_test], axis=1)

    # Prepare data and labels
    X_train = train.drop(columns=['root'])
    y_train = train['root']
    X_test = test.drop(columns=['root'])
    y_test = test['root']

else: 
    # One-hot encode 'language' in train and test (all test data) sets
    enc = OneHotEncoder(sparse_output=True)
    language_encoded = enc.fit_transform(train[['language']])
    language_df = pd.DataFrame(language_encoded.toarray(),
                            columns=enc.get_feature_names_out(['language']),
                            index=train.index)
    train = pd.concat([train.drop(columns=['language']), language_df], axis=1)

    # Same for test set
    language_encoded_test = enc.transform(test[['language']])
    language_df_test = pd.DataFrame(language_encoded_test.toarray(),
                                    columns=enc.get_feature_names_out(['language']),
                                    index=test.index)
    test = pd.concat([test.drop(columns=['language']), language_df_test], axis=1)

    #Prepare data and labels
    X_train = train.drop(columns=['root'])
    y_train = train['root']
    X_test = test

In [3]:
X_test

,sentence_id,node,degree,avg_neighbor_deg,degree_squared,degree_diff,clustering,local_degree_ratio,max_neighbor_degree,degree_centrality,...,language_Italian,language_Japanese,language_Korean,language_Polish,language_Portuguese,language_Russian,language_Spanish,language_Swedish,language_Thai,language_Turkish
0,1,38,2,2.5,4,-0.5,0,0.799997,4,0.047619,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,33,1,2.0,1,-1.0,0,0.499998,2,0.023810,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,10,4,2.0,16,2.0,0,1.999990,3,0.095238,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,24,2,1.5,4,0.5,0,1.333324,2,0.047619,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,16,1,2.0,1,-1.0,0,0.499998,2,0.023810,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194643,993,7,2,3.0,4,-1.0,0,0.666664,4,0.133333,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
194644,993,5,2,1.5,4,0.5,0,1.333324,2,0.133333,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
194645,993,11,1,2.0,1,-1.0,0,0.499998,2,0.066667,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
194646,993,2,2,2.5,4,-0.5,0,0.799997,4,0.133333,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


## Models

### Multimodel Random Forest

#### Train One Random Forest per Language

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from scipy.stats import uniform, randint
import pandas as pd
import numpy as np

# Step 1: Decode Language in Training Set
language_columns = [col for col in X_train.columns if col.startswith('language_')]
X_train['language'] = enc.inverse_transform(X_train[language_columns])[:, 0]
X_train = X_train.drop(columns=language_columns)

# Step 2: Normalize Features dividing by length of the sentence


# Step 2.5: Train One XGBoost with Hyperparameter Tuning per Language
language_models = {}

param_distributions = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'gamma': uniform(0, 1)
}

for language in X_train['language'].unique():
    print(f"\n🔤 Training model for language: {language}")
    X_lang = X_train[X_train['language'] == language].copy()
    y_lang = y_train[X_lang.index]

    X_features = X_lang.drop(columns=['language'])
    
    clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', verbosity=0)

    search = RandomizedSearchCV(
        clf,
        param_distributions=param_distributions,
        n_iter=10,
        scoring='accuracy',
        cv=3,
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_features, y_lang)
    print(f"✅ Best params for {language}: {search.best_params_}")

    language_models[language] = search.best_estimator_

# Step 3: Prepare X_test for Language-Based Prediction
language_columns = [col for col in X_test.columns if col.startswith('language_')]
X_test['language'] = enc.inverse_transform(X_test[language_columns])[:, 0]
X_test = X_test.drop(columns=language_columns)


# Step 4: Predict Using the Matching Model
X_test['root'] = 0.0  # Placeholder column for probabilities

for language, clf in language_models.items():
    lang_subset = X_test['language'] == language
    if lang_subset.any():
        X_lang_test = X_test[lang_subset].drop(columns=['language', 'root'])
        probs = clf.predict_proba(X_lang_test)[:, 1]
        X_test.loc[lang_subset, 'root'] = probs

# Step 5: Find root nodes
def find_root(group):
    return group.loc[group['root'].idxmax()]['node']

grouped = X_test.groupby(['language', 'sentence_id'])
roots = grouped.apply(find_root).reset_index(name='root')
roots.insert(0, 'id', range(1, len(roots) + 1))
roots = roots[['id', 'root']]
roots.to_csv('data/predictions_submission.csv', index=False)

# Optional Kaggle Evaluation
if TESTING_WITH_KAGGLE_DATA:
    def evaluate_model(y_true, y_pred):
        print("Print number of correct predictions:")
        correct_predictions = (y_true == y_pred).sum()
        final_score = correct_predictions / len(y_true)
        print(f"Evaluation accuracy: {final_score}")
    
    kaggle_perfect_predictions = pd.read_csv("data/kaggle_perfect_predictions.csv").drop(columns=['id']).drop(index=0)
    current_predictions = pd.read_csv('data/predictions_submission.csv').drop(columns=['id']).drop(index=0)
    print(classification_report(kaggle_perfect_predictions, current_predictions))



🔤 Training model for language: Japanese
Fitting 3 folds for each of 10 candidates, totalling 30 fits
✅ Best params for Japanese: {'colsample_bytree': np.float64(0.8401537692938899), 'gamma': np.float64(0.450499251969543), 'learning_rate': np.float64(0.013979488347959958), 'max_depth': 3, 'n_estimators': 415, 'subsample': np.float64(0.7816441089227697)}

🔤 Training model for language: Finnish
Fitting 3 folds for each of 10 candidates, totalling 30 fits
✅ Best params for Finnish: {'colsample_bytree': np.float64(0.8401537692938899), 'gamma': np.float64(0.450499251969543), 'learning_rate': np.float64(0.013979488347959958), 'max_depth': 3, 'n_estimators': 415, 'subsample': np.float64(0.7816441089227697)}

🔤 Training model for language: Galician
Fitting 3 folds for each of 10 candidates, totalling 30 fits
✅ Best params for Galician: {'colsample_bytree': np.float64(0.8059264473611898), 'gamma': np.float64(0.13949386065204183), 'learning_rate': np.float64(0.09764339456056544), 'max_depth': 9,

In [ ]:
evaluate_model(kaggle_perfect_predictions, current_predictions)

Print number of correct predictions:
Evaluation accuracy: root    0.076871
dtype: float64
